In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
# from BACKTEST.backtest import *
# from MODELS.pipeline import *
# from DATA.TOOLS.fetchPlayersStatsV2 import *
from DATA.TOOLS.fetchPlayersStats import *
from DATA.TOOLS.fetchTeamStats import *
from DATA.TOOLS.playerPositions import *

## Fetches Player Gamelogs

In [2]:
nba = FetchPlayersStats()
data = nba.getCompleteStats(
    season='2025-26', 
    season_type='Regular Season', 
    sleep_time=2, 
    max_workers=5,
    batch_limit=100,
    complete_cache_file='../DATA/CSV_FILES/REGULAR_DATA/S26.csv',
    include_playbyplay=False
)
data.tail()

Processing 9 games (limited by batch_limit=100)

Processing batch 1/1 (9 games)
Completed batch 1/1

Merging team stats...
Cache updated. Total games now: 323


,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,...,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV
7085,Brandon Williams,1630314,DAL vs. MIA,DAL,1610612742,MIA,1,0022500332,2025-12-03,W,...,102.0,108.0,40.0,101.0,0.396,41.0,29.0,11.0,5.0,11.0
7086,Nikola Jović,1631107,MIA @ DAL,MIA,1610612748,DAL,0,0022500332,2025-12-03,L,...,116.5,118.0,46.0,91.0,0.505,50.0,31.0,6.0,7.0,15.0
7087,D'Angelo Russell,1626156,DAL vs. MIA,DAL,1610612742,MIA,1,0022500332,2025-12-03,W,...,102.0,108.0,40.0,101.0,0.396,41.0,29.0,11.0,5.0,11.0
7088,Daniel Gafford,1629655,DAL vs. MIA,DAL,1610612742,MIA,1,0022500332,2025-12-03,W,...,102.0,108.0,40.0,101.0,0.396,41.0,29.0,11.0,5.0,11.0
7089,Luke Kennard,1628379,ATL vs. LAC,ATL,1610612737,LAC,1,0022500327,2025-12-03,L,...,118.8,115.0,44.0,89.0,0.494,54.0,29.0,12.0,1.0,12.0


In [3]:
pd.set_option('display.max_columns', None)

s19_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S19.csv')
s20_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S20.csv')
s21_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S21.csv')
s22_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S22.csv')
s23_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S23.csv')
s24_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S24.csv')
s25_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S25.csv')
s26_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S26.csv')

for df in [s20_regular, s21_regular, s22_regular, s23_regular, s24_regular, s25_regular, s26_regular]:
    df.drop(columns=['Unnamed: 0', 'DEF_FG_PCT_ALLOWED', 'DEF_3PT_PCT_ALLOWED', 'PTS_ALLOWED_PER_MIN', 
                     'DEF_TOV_FORCED_PER_MIN', 'DEF_BLOCKS_PER_MIN', 'DEF_SHOOTING_FOULS_PER_MIN', 
                     'DEF_AST_ALLOWED_PER_MIN'], errors='ignore', inplace=True)

## Fetching play by play data for model v2

In [4]:
# nba = FetchPlayersStats()
# data = nba.getCompleteStats(
#     season='2023-24', 
#     season_type='Regular Season', 
#     sleep_time=2, 
#     max_workers=5,
#     batch_limit=50,
#     complete_cache_file='../DATA/CSV_FILES/REGULAR_DATA/S24v2.csv',
#     include_playbyplay=True
# )
# data.tail()

## Assign features for regular season data

In [4]:
# Add this line after your current imports
from FEATURES.features import teamContext as features_teamContext

def convert_min_to_float(min_str):
    try:
        if isinstance(min_str, str) and ":" in min_str:
            minutes, seconds = map(int, min_str.split(":"))
            total_minutes = minutes + seconds / 60
            return round(total_minutes, 2)
        elif isinstance(min_str, (int, float)):
            return float(min_str)
        else:
            return 0
    except:
        return 0

# def awayGame(df):
#     df['AWAY_GAME'] = df['MATCHUP'].str.contains('@').astype(int)
#     return df

def process_season_features(season_df, prop_type, year):
    df = season_df.copy()
    df = sort_data_for_features(df)
    df['STARTING'] = df['START_POSITION'].apply(lambda x: 1 if x in ['G','F','C'] else 0)
    # df = awayGame(df)
    # df['BLOWOUT_RISK'] = (abs(df['spread']) >= 10).astype(int)
    # df['COMPETITIVE_GAME'] = (abs(df['spread']) < 5).astype(int)
    # df['TEAM_IMPLIED_PTS'] = ((df['total'] + df['team_spread']) / 2).round(1)
    
    # Add position data
    cache_file = os.path.join(project_root, 'DATA', 'TOOLS', 'playerInfo.csv')
    df = assign_position_with_cache(df, cache_file=cache_file, max_workers=4, delay_between_requests=1.5)
   
    df = add_rest_day_features(df)
    df['MIN'] = df['MIN'].apply(convert_min_to_float)
    # df, team_mapping = encode_teams_label(df)

    df = rollingAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', windows=[3,5,7,10,20])
    df = statAgainstTeam(df, player_id_col='PLAYER_ID', opp_col='OPP_ABBREVIATION')
    df = HomeAwayAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
    df = getPlayerAvgToDateVectorized(df) 
    df = features_teamContext(df)
    df = assign_opponent_team_stats_dict(df)
    df = add_volatility_features(df, windows=[10,20,40])
    df = assign_team_opp_def_by_position(df, min_minutes=1)
    df = process_star_players_data(df, min_minutes=20, min_games=5)
    df = add_performance_without_stars_columns(df, min_games=1)
    df = add_opponent_team_rolling_stats(df, windows=[3,5])
    df = expectedPace(df)
    df = add_usual_starters_availability(df)
    df = add_team_rolling_stats(df, windows=[3, 5, 7,])
    df = calculate_league_avg_team_def_rating(df)
    df = calculate_league_avg_team_pace(df)
    df = calculate_league_avg_team_off_rating(df)
    df = get_standard_deviation(df, windows=[5,10,15])
    df = add_lineup_usage_fga_share(df)
    df = add_interaction_features(df)

    # Clean up any unwanted columns
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
    if 'Unnamed: 0.1' in df.columns:
        df.drop(columns=['Unnamed: 0.1'], inplace=True)
    return df

prop_types = ['PTS']
# data = [s22_regular,s23_regular,s24_regular, s25_regular, s26_regular]
# seasons = [2022,2023,2024, 2025, 2026]
data = [s26_regular]
seasons = [2026]

# Pre-define output directory once
output_dir = os.path.join(project_root, 'DATA', 'CSV_FILES', 'TRAIN_DATA')
os.makedirs(output_dir, exist_ok=True)

# Process each season sequentially but with optimized operations
for season_data, year in zip(data, seasons):
    print(f"Processing year {year}...")
    
    # Process features
    processed_data = process_season_features(
        season_data, 
        prop_type='PTS',
        year=year
    )
    
    # Save file
    output_path = os.path.join(output_dir, f'PTS_TRAIN_{str(year)[-2:]}.csv')
    processed_data.to_csv(output_path)
    print(f"Completed {year}")

Processing year 2026...
Loading position cache...
Loaded 1244 players from cache
Found 492 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1244 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!
Completed 2026
